# Lecture 5 lab — abandonment and what else the call log can tell us

Start with the agent queue and ask what changes when callers can leave. Keep the earlier arrival input for this test. Then open the original call log and try adding one recorded detail at a time.

From the repository root, copy the notebook into `students/` and open the copy in your local Python environment.

```bash
mkdir -p students
cp notebooks/lecture5_evaluation_exercise.ipynb students/lecture5_queue_work.ipynb
```

You need the official `November1999.txt` in the Git-ignored `data/raw/` folder. Follow the [local data setup](../data/README.md#lecture-5-original-morning-records). The lab uses the Python standard library. Complete each `# TODO` before running the cell. Replace its `raise NotImplementedError(...)` with your code. The small checks compare your implementation with examples you can work out by hand. Keep the data and any output showing individual calls on your computer.

## 1. Identify the request time

| What happened | When to count the request |
| --- | --- |
| The caller joined the agent queue | Use the recorded time `q_start`, even if the caller later hung up. |
| The caller went directly to an agent | Use the recorded `ser_start` as the request time. The queue wait is zero. |
| The caller left the automated menu before joining the queue | This call contributes no agent request. |

Use Sunday–Thursday, 09:00 to before 11:00. The loader keeps one time per request, measured in seconds since 09:00. Two requests recorded in the same second are both kept. Use **1–14 November** to fit the models. Keep **15–30 November** for checking the arrival, service, and patience inputs. December is reserved for checking the combined queue in Lecture 6, our final meeting.

Five requests on the fit mornings have a queue start and the outcome label `AGENT`, but no service start. Use their recorded arrival times. Set them aside when an analysis requires a verified service outcome.

In [ ]:
import csv
import math
import random
import statistics
import sys
from bisect import bisect_left, bisect_right
from itertools import pairwise
from pathlib import Path


here = Path.cwd()
for folder in (here, *here.parents):
    if (folder / "data" / "anonymousbank_mornings.py").is_file():
        break
else:
    raise FileNotFoundError("Open the notebook from inside the course repository.")

raw_file = folder / "data" / "raw" / "November1999.txt"
if not raw_file.is_file():
    raise FileNotFoundError("Place November1999.txt in data/raw/. See data/README.md.")

sys.path.insert(0, str(folder / "data"))
from anonymousbank_mornings import MORNING_SECONDS, load_mornings


requests_by_day, services_by_day = load_mornings(raw_file)
fit_days = [day for day in sorted(requests_by_day) if day.day <= 14]
check_days = [day for day in sorted(requests_by_day) if day.day > 14]
fit_arrivals = [[call.arrival for call in requests_by_day[day]] for day in fit_days]
check_arrivals = [[call.arrival for call in requests_by_day[day]] for day in check_days]

print(f"Fit: {len(fit_days)} mornings, {sum(map(len, fit_arrivals)):,} requests")
verified_requests = sum(call.served is not None for day in fit_days for call in requests_by_day[day])
print(f"Requests with a verified fit-period outcome: {verified_requests:,}")

## 2. Estimate how long callers are willing to wait

**Patience** is how long a caller would wait before leaving the queue. A caller who abandons after 40 seconds reveals a patience time of 40 seconds in the model. A caller served after 25 seconds was still willing to wait at 25 seconds. We only know a lower bound for that caller's patience. This is a **right-censored** observation.

Use the fit mornings and the individual recorded waiting times. Assume independent exponential patience with a constant rate $\theta$ per second. A caller still waiting has probability approximately $\theta\Delta t$ of abandoning over a short interval of length $\Delta t$. Estimate the rate by dividing the **number of abandonments** by the **total observed waiting seconds**, including waits that ended in service. The assumed time at which an agent becomes available is independent of the caller's patience.

**Task 1.** Complete `fit_patience_rate`. If three callers spend a total of 60 seconds waiting and one abandons, the estimated rate is $1/60$ per second. What would go wrong if you added up only the waits of callers who abandoned?

In [ ]:
def fit_patience_rate(abandonments, total_wait_seconds):
    # TODO (Task 1): return the fitted rate per second.
    raise NotImplementedError("Complete Task 1")


assert math.isclose(fit_patience_rate(1, 60), 1 / 60)
observed_by_day = [
    [(call.served, call.wait) for call in requests_by_day[day] if call.served is not None]
    for day in fit_days
]
observed = [record for morning in observed_by_day for record in morning]
abandonments = sum(not served for served, _ in observed)
waiting_seconds = sum(wait for _, wait in observed)
patience_rate = fit_patience_rate(abandonments, waiting_seconds)
print(f"{abandonments:,} abandonments and {waiting_seconds:,} observed waiting seconds")
print(f"Mean patience from the fit: {1 / patience_rate:.1f} seconds")

### Does the abandonment rate change as callers wait?

Estimate the rate for each of the first four minutes and for waits of four minutes or more. A caller who abandons after 80 seconds contributes 60 waiting seconds to minute one, 20 to minute two, and one abandonment in minute two. A caller served after 25 seconds contributes 25 waiting seconds to minute one. An abandonment exactly at a minute mark goes in the interval starting at that mark.

Compare one, two, three, and five rates. Fit each candidate on 1–14 November, then score it on 15–30 November. The score is the **log likelihood** of the later records. Every observed wait contributes the log probability of remaining patient up to that time. An abandonment also contributes the log of the rate in effect when the caller left. Higher scores mean the model better accounts for those later waits and abandonment times.

**Task 1b.** Complete `waiting_by_period`. For each caller, add the number of seconds they spent waiting inside each interval. The small check below combines the 80-second abandonment and the 25-second wait.

In [ ]:
def waiting_by_period(outcomes, cutoffs):
    edges = (0, *cutoffs, math.inf)
    events = [0] * (len(edges) - 1)
    exposure = [0] * len(events)
    for served, wait in outcomes:
        for i, (start, end) in enumerate(zip(edges[:-1], edges[1:], strict=True)):
            # TODO (Task 1b): add this caller's waiting seconds inside [start, end).
            raise NotImplementedError("Complete Task 1b")
        if not served:
            events[bisect_right(cutoffs, wait)] += 1
    return events, exposure


assert waiting_by_period([(False, 80), (True, 25)], (60, 120)) == ([0, 1, 0], [85, 20, 0])

candidate_cutoffs = {
    "One rate": (),
    "Two rates": (60,),
    "Three rates": (60, 120),
    "Five rates": (60, 120, 180, 240),
}
fitted_rates = {}
validation_scores = {}
patience_check = [
    (call.served, call.wait)
    for day in check_days
    for call in requests_by_day[day]
    if call.served is not None
]
for label, cutoffs in candidate_cutoffs.items():
    fit_events, fit_exposure = waiting_by_period(observed, cutoffs)
    rates = [fit_patience_rate(events, time)
             for events, time in zip(fit_events, fit_exposure, strict=True)]
    fitted_rates[label] = rates

    check_events, check_exposure = waiting_by_period(patience_check, cutoffs)
    validation_scores[label] = sum(
        events * math.log(rate) - time * rate
        for events, time, rate in zip(check_events, check_exposure, rates, strict=True)
    )

minutes = ("Minute 1", "Minute 2", "Minute 3", "Minute 4", "From 4 min")
events, exposure = waiting_by_period(observed, candidate_cutoffs["Five rates"])
print(f"{'Time since joining':<20} {'Abandonments':>13} {'Waiting seconds':>16} {'Rate per minute':>16}")
for label, count, time, rate in zip(minutes, events, exposure, fitted_rates["Five rates"], strict=True):
    print(f"{label:<20} {count:13,} {time:16,} {60 * rate:16.3f}")

baseline = validation_scores["One rate"]
print(f"\nValidation data: {len(patience_check):,} requests with verified outcomes")
print(f"{'Model':<20} {'Score gain over one rate':>25}")
for label, score in validation_scores.items():
    print(f"{label:<20} {score - baseline:25.1f}")

**Exercise. Which patience model would you use?** Before continuing, write two or three sentences using both tables. Explain where you would let the abandonment rate change and whether the improvement in validation score justifies adding more rates.

## 3. Simulate a morning with a common queue

Start the simulated morning with free agents at 09:00. Repeat the Lecture 4 lognormal fit on the individual service durations recorded on 1–14 November. Use services starting from 09:00 to before 11:00 and keep durations of at least 10 seconds. Share a single queue across the agents and serve calls in order of arrival. The bank used priorities and separate routing for some service types. The simplified queue also assumes the same number of agents throughout the morning.

Use **Model A**, the arrival generator from the main Week 3 and 4 experiments. It draws one rate for each morning and generates exponential gaps at that rate until 11:00. Fit its parameters to requests for agents from 1–14 November. Give all three patience rules the same simulated arrivals and service durations. Generate one service time per call. Reuse arrivals and service durations when you compare the different patience rules.

**Task 2.** Complete the service decision. If all agents are busy, calculate when the earliest agent could take this caller. The caller abandons if that wait would exceed their patience. In that case record the patience time as the wait and leave the agent's schedule alone. If service can begin at the patience deadline, the caller is served.

**Check by hand.** In the first example below, caller 1 occupies the agent from 0 to 80 seconds. Caller 2 arrives at 10 and would have to wait 70 seconds. Their patience is 20 seconds. What happens to caller 3, who arrives at 80?

In [ ]:
def simulate_queue(arrivals, services, patience, agents):
    free_at = [0.0] * agents
    records = []
    for arrival, duration, limit in zip(arrivals, services, patience, strict=True):
        agent = min(range(agents), key=free_at.__getitem__)
        start = max(arrival, free_at[agent])
        wait = start - arrival
        # TODO (Task 2): append (served, observed_wait) and reserve the agent only if served.
        raise NotImplementedError("Complete Task 2")
    return records


assert simulate_queue([0, 10, 80], [80, 500, 10], [10, 20, 5], 1) == [
    (True, 0), (False, 20), (True, 0),
]
assert simulate_queue([0, 5], [10, 10], [5, 5], 1) == [(True, 0), (True, 5)]

## 4. What changes when callers can leave?

Use nine agents throughout the morning as a reference scenario. The observed request rate is 3,115 calls over 1,200 minutes, or about 2.60 calls per minute. The fitted mean service time is 3.43 minutes. Their product is about 8.9 agent minutes of work arriving per minute. The call logs show 10–13 agents handling a call on each of the ten fit mornings. The daily peak of agents on calls at once ranges from 9 to 12 and exceeds nine on nine mornings. This makes nine a low-staffing scenario. A roster would also identify available agents who were idle.

Use the three-rate fit for the shared simulation comparison. Run 600 mornings with each patience rule. Share arrivals and service durations. For the two patience models, share a random exponential draw with mean one. With a constant rate, divide this draw by the rate. With three rates, each completed minute uses up 60 times its rate from the draw. When the remainder fits in an interval, divide it by that interval's rate to get the additional wait within that interval. This gives each caller one patience time under each model.

Calculate the outcomes for each morning first, then give each morning equal weight. The abandonment share and the share served within 60 seconds both divide by all requests with a known outcome. The mean served wait averages only over served callers, including zero waits.

**Task 3.** Complete the two counts in `morning_outcomes`: how many callers abandoned, and how many were served within 60 seconds. Count only served callers for the second measure and include a wait of exactly 60 seconds. Then explain why allowing callers to hang up can raise the share served within a minute even though fewer callers eventually get served. Compare the constant-rate fit with the selected three-rate fit on abandonment and service within one minute. Use the observed November outcomes as a reference and identify a mismatch to check next. These same dates supplied the inputs. Use the gap between the observations and the simulation to motivate looking at more fields from the call log.

In [ ]:
selected_cutoffs = candidate_cutoffs["Three rates"]
selected_rates = fitted_rates["Three rates"]


QUARTER_HOUR = 15 * 60


def quarter_hour_counts(arrivals):
    counts = [0] * (MORNING_SECONDS // QUARTER_HOUR)
    for time in arrivals:
        counts[time // QUARTER_HOUR] += 1
    return counts


fit_counts = [sum(quarter_hour_counts(morning)) for morning in fit_arrivals]
mean_count = statistics.mean(fit_counts)
base_rate = mean_count / MORNING_SECONDS
# As in Week 3, subtract the count variation already explained by Poisson arrivals.
alpha = max(0.0, (statistics.variance(fit_counts) - mean_count) / mean_count**2)


def generate_parametric_morning(rng, horizon=MORNING_SECONDS, *, mean_rate=base_rate, daily_factor=True):
    # Draw the factor in both variants so the subsequent gap draws line up.
    multiplier = rng.gammavariate(1 / alpha, alpha) if alpha > 0 else 1.0
    rate = mean_rate * (multiplier if daily_factor else 1.0)
    arrivals = []
    clock = 0.0
    while True:
        clock += rng.expovariate(rate)
        if clock >= horizon:
            return arrivals
        arrivals.append(clock)


arrival_rng = random.Random(2029)
parametric_arrivals = []
arrival_rng_states = []
for _ in range(600):
    # Save the random generator's position so later variants use the same draws.
    arrival_rng_states.append(arrival_rng.getstate())
    parametric_arrivals.append(generate_parametric_morning(arrival_rng))

def fit_lognormal(durations):
    mean = statistics.mean(durations)
    variance = statistics.pvariance(durations)
    log_variance = math.log1p(variance / mean**2)
    return math.log(mean) - log_variance / 2, math.sqrt(log_variance)


fit_services = [service for day in fit_days for service in services_by_day[day]]
pool_parameters = fit_lognormal([service.duration for service in fit_services])


def service_times(normals, parameters):
    log_mean, log_sd = parameters
    return [math.exp(log_mean + log_sd * draw) for draw in normals]


def piecewise_patience(draw, cutoffs, rates):
    start = 0
    for end, rate in zip(cutoffs, rates[:-1], strict=True):
        interval_hazard = (end - start) * rate
        if draw <= interval_hazard:
            return start + draw / rate
        draw -= interval_hazard
        start = end
    return start + draw / rates[-1]


def draw_service_and_patience(count, seed):
    rng = random.Random(seed)
    normals, draws = [], []
    for _ in range(count):
        normals.append(rng.normalvariate(0, 1))
        draws.append(rng.expovariate(1.0))
    limits = {
        "No abandonment": [math.inf] * count,
        "One patience rate": [draw / patience_rate for draw in draws],
        "Three patience rates": [piecewise_patience(draw, selected_cutoffs, selected_rates) for draw in draws],
    }
    return normals, limits


def morning_outcomes(records):
    waits = [wait for served, wait in records if served]
    n = len(records)
    # TODO (Task 3): calculate abandoned and served_within_60.
    raise NotImplementedError("Complete Task 3")
    return {
        "abandonment": abandoned / n,
        "served_in_60": served_within_60 / n,
        "mean_served_wait": statistics.mean(waits),
    }


assert morning_outcomes([(True, 0), (True, 60), (True, 61), (False, 20)]) == {
    "abandonment": 0.25,
    "served_in_60": 0.50,
    "mean_served_wait": 121 / 3,
}


seed_rng = random.Random(2027)
draw_seeds = [seed_rng.getrandbits(64) for _ in parametric_arrivals]
parametric_draws = [
    draw_service_and_patience(len(morning), seed)
    for morning, seed in zip(parametric_arrivals, draw_seeds, strict=True)
]
runs = {label: [] for label in ("No abandonment", "One patience rate", "Three patience rates")}
for arrivals, (normals, limits) in zip(parametric_arrivals, parametric_draws, strict=True):
    services = service_times(normals, pool_parameters)
    for label, patience in limits.items():
        records = simulate_queue(arrivals, services, patience, agents=9)
        runs[label].append(morning_outcomes(records))


def report_outcomes(label, mornings):
    averages = {key: statistics.mean(day[key] for day in mornings) for key in mornings[0]}
    print(f"{label:<25} {averages['abandonment']:>10.1%} "
          f"{averages['served_in_60']:>14.1%} {averages['mean_served_wait']:>17.1f}")


print(f"{'Nine-agent scenario':<25} {'Abandoned':>10} {'Served <=60 s':>14} {'Served wait (s)':>17}")
report_outcomes("Observed 1–14 Nov", [morning_outcomes(day) for day in observed_by_day])
for label, mornings in runs.items():
    report_outcomes(label, mornings)

## 5. Inspect the original call log

The queue still misses some observed outcomes. Which fields could explain that? Read a few rows of the original tab-separated file locally. The cell covers all 17 columns and masks the identifiers. See the [data guide](../data/anonymousbank_data_guide.md#what-one-raw-record-contains) for the meanings of the source codes.

| What the field tells us | Source columns |
| --- | --- |
| Calendar date | `date` |
| The call, caller, and phone-system line | `call_id`, `customer_id`, `vru+line` |
| Phone menu entry, exit, and time spent there | `vru_entry`, `vru_exit`, `vru_time` |
| Category of service and priority | `type`, `priority` |
| Queue entry, exit, and time spent waiting | `q_start`, `q_exit`, `q_time` |
| Agent service start, end, duration, and agent identifier | `ser_start`, `ser_exit`, `ser_time`, `server` |
| Recorded call outcome | `outcome` |

Point to the fields you would need to change (1) when requests arrive, (2) how long service takes, or (3) who moves to the front of the queue. Which fields are known only after a call has ended?

In [ ]:
private_columns = {"call_id", "customer_id", "vru+line", "server"}
with raw_file.open(encoding="latin-1", newline="") as source:
    reader = csv.DictReader(source, delimiter="	")
    print(", ".join(reader.fieldnames))
    shown = 0
    for row in reader:
        if row["outcome"].strip() == "PHANTOM":
            continue
        print()
        for field in reader.fieldnames:
            value = "[masked]" if field in private_columns else row[field].strip()
            print(f"{field:>12}  {value}")
        shown += 1
        if shown == 2:
            break

## 6. Fit exponential gaps from individual timestamps

Subtract each recorded request time from the next request time on the same morning. For example, 09:00:12, 09:00:36, and 09:01:34 give gaps of 24 and 58 seconds. Keep zero gaps when two requests were recorded in the same second. Start at the first recorded request on each morning.

Fit an exponential distribution to the complete gaps. Set its mean to the sample mean of those gaps. The fitted rate is the reciprocal of that mean.

Next compare two ways of using that fit. One uses the same rate every morning. The other multiplies it by a fresh daily factor, as in Week 3. Reuse the earlier Gamma factor fit and use the same underlying random draws across the exponential variants. This comparison changes one assumption at a time.

**Task 4.** Complete `interarrival_gaps` using consecutive pairs from each morning. Return one list of gaps, including zeros. Then check the fitted rate against the earlier count-based rate. On later November mornings, compare the mean and standard deviation of the gaps and the standard deviation of the total requests per morning. Which measure shows the effect of adding a daily factor most clearly?

In [ ]:
def interarrival_gaps(mornings):
    gaps = []
    for morning in mornings:
        for earlier, later in pairwise(morning):
            # TODO (Task 4): append the gap between the two timestamps.
            raise NotImplementedError("Complete Task 4")
    return gaps


assert interarrival_gaps([[0, 0, 60], [5, 10], [4]]) == [0, 60, 5]


fit_gaps = interarrival_gaps(fit_arrivals)
mean_gap = statistics.mean(fit_gaps)
timestamp_rate = 1 / mean_gap
print(f"Completed gaps: {len(fit_gaps):,}")
print(f"Fitted mean gap: {mean_gap:.2f} seconds")
print(f"Rate from timestamps: {60 * timestamp_rate:.3f} requests/min")
print(f"Earlier rate from counts: {60 * base_rate:.3f} requests/min")


def simulate_gap_fit(daily_factor):
    rng = random.Random()
    mornings = []
    for state in arrival_rng_states:
        rng.setstate(state)
        mornings.append(generate_parametric_morning(rng, mean_rate=timestamp_rate, daily_factor=daily_factor))
    return mornings


gap_arrivals = simulate_gap_fit(daily_factor=False)
gap_daily_arrivals = simulate_gap_fit(daily_factor=True)


def show_gap_check(label, mornings, *, simulated=False):
    if simulated:
        # Compare simulated times at the call log's one-second resolution.
        mornings = [[int(time) for time in morning] for morning in mornings]
    gaps = interarrival_gaps(mornings)
    print(f"{label:<24} {statistics.mean(gaps):>12.1f} {statistics.stdev(gaps):>10.1f} "
          f"{statistics.stdev(map(len, mornings)):>12.1f}")


print()
print(f"{'Arrival model':<24} {'Mean gap (s)':>12} {'Gap SD (s)':>10} {'Count SD':>12}")
show_gap_check("Observed 15–30 Nov", check_arrivals)
show_gap_check("A: Counts + daily factor", parametric_arrivals, simulated=True)
show_gap_check("A: Gaps, fixed rate", gap_arrivals, simulated=True)
show_gap_check("A: Gaps + daily factor", gap_daily_arrivals, simulated=True)


### Repeat the earlier resampling comparison

**Model B** draws a fit morning at random, with replacement. Generate one version from its 15-minute counts and one from its second-resolution timestamps.

- **From counts:** keep the number of arrivals in each quarter-hour. Generate that many uniform times inside the interval, as before.
- **From timestamps:** use that morning's recorded times in order, including requests in the same second.

Both versions use the same sampled mornings and quarter-hour totals. Use the earlier count-based Model A as the reference in this table.

The table reports the share of all requests that have occurred before each clock time and the average busiest minute. To find each morning's busiest minute, take the maximum count in any 60-second window at whole-second resolution. When the two resampling versions have the same half-hour shares, what can still differ in their busiest minutes?

In [ ]:
def arrival_times_from_counts(counts, rng):
    arrivals = []
    for quarter, count in enumerate(counts):
        start = quarter * QUARTER_HOUR
        arrivals.extend(start + QUARTER_HOUR * rng.random() for _ in range(count))
    return sorted(arrivals)


morning_rng = random.Random(2030)
sampled_mornings = [requests_by_day[morning_rng.choice(fit_days)] for _ in parametric_arrivals]
timestamp_arrivals = [[call.arrival for call in morning] for morning in sampled_mornings]
count_rng = random.Random(2028)
count_arrivals = [
    arrival_times_from_counts(quarter_hour_counts(morning), count_rng)
    for morning in timestamp_arrivals
]


def busiest_minute(arrivals):
    # Compare recorded and simulated arrivals at the same clock resolution.
    seconds = [int(time) for time in arrivals]
    left = peak = 0
    for right, time in enumerate(seconds):
        while time - seconds[left] >= 60:
            left += 1
        peak = max(peak, right - left + 1)
    return peak


assert busiest_minute([0, 10, 60, 60]) == 3


def show_timing(label, mornings):
    total = sum(map(len, mornings))
    shares = [
        sum(bisect_left(morning, cutoff) for morning in mornings) / total
        for cutoff in (30 * 60, 60 * 60, 90 * 60)
    ]
    mean_peak = statistics.mean(busiest_minute(morning) for morning in mornings)
    print(f"{label:<22} {shares[0]:>8.1%} {shares[1]:>8.1%} {shares[2]:>8.1%} {mean_peak:>15.2f}")


print()
print(f"{'Requests before':<22} {'09:30':>8} {'10:00':>8} {'10:30':>8} {'Busiest minute':>15}")
show_timing("Observed 15–30 Nov", check_arrivals)
show_timing("A: Counts + daily factor", parametric_arrivals)
show_timing("B: Resample counts", count_arrivals)
show_timing("B: Resample timestamps", timestamp_arrivals)


### Does timestamp resolution change the staffing comparison?

Run nine and ten agents under the exponential variants and both versions of the resampling model. The two resampling runs in each pair keep the same number of requests and use the same service and patience draws. Assign the draws in arrival order. Only the times within the quarter-hours change. An exponential variant can generate a different number of calls from the resampled morning. Each model reuses service and patience random draws by arrival order.

**Task 5.** Complete `staffing_gains` by pairing the nine- and ten-agent outcomes for each morning. Return the list of differences in the same order as the mornings. A positive difference means the extra agent increased the share served within a minute. At each staffing level, compare the simulated share served within a minute. How much does adding the daily factor change the exponential model's result? How does the estimated gain from adding an agent change when you use timestamps in the resampling model? Use the paired Monte Carlo interval from Lecture 3 to judge the uncertainty on that difference. Would the staffing choice change for a manager targeting an average of at least 75% served within a minute? This is a hypothetical target under the stated staffing and routing assumptions.

In [ ]:
def simulate_staffing(arrival_paths, durations_by_morning, draws_by_morning):
    outcomes = {9: [], 10: []}
    for arrivals, services, (_, limits) in zip(
        arrival_paths, durations_by_morning, draws_by_morning, strict=True
    ):
        for agents in outcomes:
            records = simulate_queue(arrivals, services, limits["Three patience rates"], agents)
            outcomes[agents].append(morning_outcomes(records))
    return outcomes


def service_and_patience_for(mornings):
    return [draw_service_and_patience(len(morning), seed)
            for morning, seed in zip(mornings, draw_seeds, strict=True)]


def staffing_with_pooled_service(mornings, draws):
    durations = [service_times(normals, pool_parameters) for normals, _ in draws]
    return simulate_staffing(mornings, durations, draws)


bootstrap_draws = service_and_patience_for(timestamp_arrivals)
staffing_runs = {
    "A: Counts + daily factor": staffing_with_pooled_service(parametric_arrivals, parametric_draws),
    "A: Gaps, fixed rate": staffing_with_pooled_service(gap_arrivals, service_and_patience_for(gap_arrivals)),
    "A: Gaps + daily factor": staffing_with_pooled_service(gap_daily_arrivals, service_and_patience_for(gap_daily_arrivals)),
    "B: Resample counts": staffing_with_pooled_service(count_arrivals, bootstrap_draws),
    "B: Resample timestamps": staffing_with_pooled_service(timestamp_arrivals, bootstrap_draws),
}


def staffing_gains(runs):
    nine = [day["served_in_60"] for day in runs[9]]
    ten = [day["served_in_60"] for day in runs[10]]
    gains = []
    for before, after in zip(nine, ten, strict=True):
        # TODO (Task 5): append the ten-agent share minus the nine-agent share.
        raise NotImplementedError("Complete Task 5")
    return statistics.mean(nine), statistics.mean(ten), gains


example_runs = {
    9: [{"served_in_60": 0.25}, {"served_in_60": 0.75}],
    10: [{"served_in_60": 0.50}, {"served_in_60": 0.75}],
}
assert staffing_gains(example_runs) == (0.50, 0.625, [0.25, 0.0])


def report_staffing(model_runs):
    print(f"{'Model input':<24} {'9 agents':>10} {'10 agents':>10} {'Gain (pp)':>10}")
    gains_by_model = {}
    for label, runs in model_runs.items():
        nine, ten, gains = staffing_gains(runs)
        gains_by_model[label] = gains
        print(f"{label:<24} {nine:>10.1%} {ten:>10.1%} {100 * statistics.mean(gains):>10.1f}")
    return gains_by_model


gains_by_model = report_staffing(staffing_runs)

def report_gain_difference(first_gains, second_gains):
    differences = [
        100 * (first - second)
        for first, second in zip(first_gains, second_gains, strict=True)
    ]
    estimate = statistics.mean(differences)
    margin = 1.96 * statistics.stdev(differences) / math.sqrt(len(differences))
    print(f"Difference in gains: {estimate:.2f} percentage points")
    print(f"Approx. 95% Monte Carlo interval: [{estimate - margin:.2f}, {estimate + margin:.2f}] pp")


print("Within Model B: timestamps minus counts")
report_gain_difference(gains_by_model["B: Resample timestamps"], gains_by_model["B: Resample counts"])


## 7. Second refinement: use the recorded service type

Lecture 4 showed that service durations differ across types of calls. Fit the same lognormal model once to all retained durations and once for each of the six recorded values of `type`. Use 1–14 November for the fits. Compare the predicted mean duration with 15–30 November for each type. Pay attention to how few observations we have for some types.

**Task 6.** Complete `service_time_by_type`. Look up the fitted lognormal parameters for the request's service type. Transform the supplied standard-normal draw into a service duration in seconds using the same rule as `service_times`. Then compare the models with the resampled timestamps, staffing, patience, and random draws held fixed. Changing the service model changes how that draw maps to seconds. This experiment assumes the service category is available when the caller requests an agent.

Which service types have large differences in mean duration? Does using the category improve the service-duration check? How much does it change the estimated gain from a tenth agent?

In [ ]:
type_labels = {
    "PS": "Regular banking",
    "PE": "Banking in English",
    "IN": "Internet consulting",
    "NE": "Stock-exchange activity",
    "NW": "Prospective customer information",
    "TT": "Returned call put on hold",
}


def durations_of_type(services, service_type):
    return [service.duration for service in services if service.service_type == service_type]


check_services = [service for day in check_days for service in services_by_day[day]]
type_parameters = {}
print(f"{'Type':<6} {'Fit n':>6} {'Fit mean (s)':>13} {'Check n':>8} {'Check mean (s)':>15}")
for service_type, description in type_labels.items():
    fitted = durations_of_type(fit_services, service_type)
    checked = durations_of_type(check_services, service_type)
    type_parameters[service_type] = fit_lognormal(fitted)
    print(f"{service_type:<6} {len(fitted):>6} {statistics.mean(fitted):>13.1f} "
          f"{len(checked):>8} {statistics.mean(checked):>15.1f}  {description}")
print(f"Pooled fitted mean: {statistics.mean(service.duration for service in fit_services):.1f} s")


def service_time_by_type(service_type, draw, parameters):
    # TODO (Task 6): select this type's parameters and convert the draw to seconds.
    raise NotImplementedError("Complete Task 6")


example_parameters = {"PS": (math.log(120), 0.5), "IN": (math.log(90), math.log(2))}
assert math.isclose(service_time_by_type("PS", 0, example_parameters), 120)
assert math.isclose(service_time_by_type("IN", 1, example_parameters), 180)


durations_by_type = [
    [service_time_by_type(call.service_type, draw, type_parameters)
     for call, draw in zip(morning, normals, strict=True)]
    for morning, (normals, _) in zip(sampled_mornings, bootstrap_draws, strict=True)
]

service_runs = {
    "Pooled service": staffing_runs["B: Resample timestamps"],
    "Service by type": simulate_staffing(timestamp_arrivals, durations_by_type, bootstrap_draws),
}
print()
gains_by_service = report_staffing(service_runs)
print("Service by type minus pooled service")
report_gain_difference(gains_by_service["Service by type"], gains_by_service["Pooled service"])

## 8. Explore the other fields

Try one of A–D in your group. Share the evidence you find with another group. Use the fit dates to propose a change and the later November dates to check the pattern. Record a specific outcome you would compare before and after changing the model.

| Option | What to check in the original log | Possible next experiment |
| --- | --- | --- |
| **A. Date and time of day** | Use `date` and the exact request times. Compare morning totals and cumulative requests by time of day for different weekdays. How many mornings support each estimate? | Resample mornings within comparable days if the difference persists on the later dates. |
| **B. The phone menu** | Use `vru_entry`, `vru_exit`, `vru_time`, `q_start`, `ser_start` and `outcome`. How long do callers spend in the menu, and what share reach an agent or its queue? | Compare a model that starts at agent requests with one that also simulates the menu and the decision to join the queue. |
| **C. Priority in the queue** | For callers with a queue start, compare `q_time` and `outcome` by `priority`. Check whether callers with different priorities also requested different `type`s of service. | Compare the first-come queue with a priority rule after checking how the bank applied priority. |
| **D. Agent activity** | Use `server`, `ser_start` and `ser_exit` locally. Count how many distinct agents were on calls and find the busiest second on each later morning. Count each agent once at each time. | Compare the daily peaks with those on fit mornings and with the staffing levels used in the model. A roster would also tell us which idle agents were available. |

Use `call_id`, `customer_id` and `vru+line` for record checks when needed. For example, check whether repeated customers or a single phone-system line account for a data pattern. Keep all identifiers and individual records local. The recorded `outcome` helps fit and evaluate the queue. It is learned after the request arrives.

The starter cell reads the fit and check dates locally for your investigation. Print summaries of your results and save your own notebook under `students/`.

In [ ]:
from datetime import datetime

with raw_file.open(encoding="latin-1", newline="") as source:
    fit_rows, check_rows = [], []
    for row in csv.DictReader(source, delimiter="	"):
        day = datetime.strptime(row["date"], "%y%m%d").date()
        if day.weekday() in (4, 5) or row["outcome"].strip() == "PHANTOM":
            continue
        (fit_rows if day.day <= 14 else check_rows).append(row)

# Start with the fields and time window for your option.
# Record one summary for fit_rows and the same summary for check_rows.

## 9. Prepare the final case for Lecture 6

Bring three notes from this notebook to the final meeting.

1. State the arrival, service, and patience models you would carry forward.
2. Name one change that mattered in the November comparisons and one mismatch the combined model needs to explain.
3. Choose a measure to check on December mornings and say how it bears on the staffing decision.

In Lecture 6, we will check the combined queue against December records, compare one staffing change with paired simulations, and review the project. Use the evidence to state a decision or identify the measurement that would help resolve it.

If you use an LLM, give it the question and your aggregate summary. Ask what other process could produce the same pattern and what check would distinguish it. Verify the suggestion on the records locally and keep individual calls and student work on your computer.